# NOEMA · continuar la auto-mejora
Sube el ZIP actualizado. Una celda instala, entrena, evalúa y descarga resultados. La ejecución medida incluida es CPU; Colab no se ejecutó desde este entorno.

In [ ]:
# Ejecuta esta única celda y selecciona ASI.zip.
GENERATIONS = 3
DEVICE = "cpu"  # Puedes usar "cuda" en una sesión con GPU.
CONTINUE_TRAINED = True

from google.colab import files
from pathlib import Path
import subprocess, sys, zipfile, tempfile, shutil

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Selecciona el ZIP del proyecto")
work = Path(tempfile.mkdtemp(prefix="noema_"))
with zipfile.ZipFile(zip_name) as archive:
    for item in archive.infolist():
        target = (work / item.filename).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError("Ruta no válida en el ZIP")
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Enlace no permitido en el ZIP")
    archive.extractall(work)
project = next(work.rglob("run_rsi.py")).parent
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(project)])
output = work / "resultados"
command = [sys.executable, str(project / "run_rsi.py"), "--generations", str(GENERATIONS),
           "--device", DEVICE, "--output", str(output)]
checkpoint = project / "examples/trained_run/checkpoint.pt"
if CONTINUE_TRAINED and checkpoint.exists():
    command += ["--resume", str(checkpoint)]
subprocess.check_call(command, cwd=project)
print((output / "report.json").read_text()[:2200])
archive_path = shutil.make_archive(str(work / "NOEMA_resultados"), "zip", output)
files.download(archive_path)
